# 19 Window Functions

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Compute values across related rows without collapsing them: rankings (<code>row_number</code>, <code>rank</code>, <code>dense_rank</code>), previous and next values (<code>lag</code>, <code>lead</code>), running totals and moving averages with frames, and top-N per group. Understand partitioning, ordering, frames, and the default-frame trap.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Window functions solve problems that <code>groupBy</code> can't, because they keep every row: deduplication (latest record per key), top-N per category, period-over-period change, running balances, sessionization and gap detection. They appear in almost every data engineering interview, often as a live coding question. This lesson is <b>interview-critical</b>.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Anatomy of a window</b><br>
A window function is computed <b>for each row</b> over a set of related rows defined by a <code>Window</code> spec:
<ul><li><code>partitionBy(...)</code>: which rows belong together (like a group, but rows are not collapsed)</li><li><code>orderBy(...)</code>: the order inside each partition (needed for ranking, lag/lead and running totals)</li><li>frame <code>rowsBetween(start, end)</code> / <code>rangeBetween(start, end)</code>: which rows around the current one are included</li></ul>
Function families:
<ul><li><b>Ranking:</b> <code>row_number</code>, <code>rank</code>, <code>dense_rank</code>, <code>percent_rank</code>, <code>ntile</code></li><li><b>Offset:</b> <code>lag</code>, <code>lead</code></li><li><b>Aggregate over a window:</b> <code>sum</code>, <code>avg</code>, <code>min</code>, <code>max</code>, <code>count</code>, <code>first</code>, <code>last</code></li></ul>
</div>

```
groupBy("dept").agg(max("salary"))         window max("salary").over(partitionBy("dept"))

dept  max                                   name   dept  salary  dept_max
Data  150000          <- 1 row per group    Amal   Data  21000   150000
Sales  39000                                Sen    Data  150000  150000   <- every row kept
                                            Vivek  Sales 25000   39000
                                            Sara   Sales 39000   39000
```

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A subscription business has 200 million billing events. For each customer it needs: the latest plan (deduplication), the change in monthly spend versus last month (<code>lag</code>), a running lifetime value (running <code>sum</code>), a 3-month moving average, and the top 3 customers by revenue in each country (<code>dense_rank</code>). All of that is window functions, partitioned by customer or country, in a single Silver-to-Gold job.
</div>

## Syntax

```python
from pyspark.sql import functions as F, Window

w = Window.partitionBy("dept").orderBy(F.desc("salary"))
df.withColumn("rn", F.row_number().over(w))
df.withColumn("prev", F.lag("amount", 1).over(Window.partitionBy("cust").orderBy("month")))

running = Window.partitionBy("cust").orderBy("month").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df.withColumn("running_total", F.sum("amount").over(running))

moving = Window.partitionBy("cust").orderBy("month").rowsBetween(-2, 0)   # current + 2 previous rows
df.withColumn("moving_avg", F.avg("amount").over(moving))
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates monthly revenue per customer and an employee table with salary ties.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A revenue table with 12 rows (3 customers × several months) and an employee table with 7 rows, where two Data employees earn the same salary.
</div>

In [0]:
from pyspark.sql import functions as F, Window

revenue = spark.createDataFrame(
    [
        ("C1", "IN", "2024-01", 100.0), ("C1", "IN", "2024-02", 120.0), ("C1", "IN", "2024-03", 90.0),
        ("C1", "IN", "2024-04", 150.0), ("C2", "IN", "2024-01", 300.0), ("C2", "IN", "2024-02", 280.0),
        ("C2", "IN", "2024-04", 310.0), ("C3", "UK", "2024-01", 50.0), ("C3", "UK", "2024-02", 75.0),
        ("C3", "UK", "2024-03", 75.0), ("C3", "UK", "2024-04", 40.0), ("C3", "UK", "2024-05", 95.0),
    ],
    "customer_id STRING, country STRING, month STRING, amount DOUBLE",
)

employees = spark.createDataFrame(
    [
        ("Amal", "Data", 90000), ("Senthil", "Data", 150000), ("Ravi", "Data", 90000),
        ("Meena", "Data", 70000), ("John", "Finance", 48000), ("Priya", "Finance", 52000),
        ("Vivek", "Sales", 39000),
    ],
    "name STRING, dept STRING, salary INT",
)
revenue.show(); employees.show()

## 1. Window aggregates keep every row

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds the department maximum, average and headcount to every employee row, and the share of the department's payroll.<br><br>
<b>Why it matters</b><br>This is the key difference from <code>groupBy</code>: you get group-level values <b>next to</b> row-level values, which makes comparisons such as "% of department total" a single expression.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
7 rows. Every Data row shows <code>dept_max</code> 150000 and <code>dept_count</code> 4. <code>pct_of_dept</code> adds up to 100 within each department.
</div>

In [0]:
by_dept = Window.partitionBy("dept")

employees.select(
    "name", "dept", "salary",
    F.max("salary").over(by_dept).alias("dept_max"),
    F.round(F.avg("salary").over(by_dept), 0).alias("dept_avg"),
    F.count("*").over(by_dept).alias("dept_count"),
    F.round(F.col("salary") / F.sum("salary").over(by_dept) * 100, 1).alias("pct_of_dept"),
).orderBy("dept", F.desc("salary")).show()

## 2. row_number, rank and dense_rank

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Ranks employees by salary within each department with the three ranking functions.<br><br>
<b>Why it matters</b><br>They differ only for <b>ties</b>, and that difference is a guaranteed interview question:
<ul><li><code>row_number</code>: 1, 2, 3, 4. Always unique, ties broken arbitrarily</li><li><code>rank</code>: 1, 2, 2, 4. Ties share a rank and leave a <b>gap</b></li><li><code>dense_rank</code>: 1, 2, 2, 3. Ties share a rank with <b>no gap</b></li></ul>
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
In Data, Amal and Ravi (90000) get <code>row_number</code> 2 and 3 (in some order), <code>rank</code> 2 and 2, <code>dense_rank</code> 2 and 2. Meena gets <code>rank</code> 4 but <code>dense_rank</code> 3.
</div>

In [0]:
by_dept_salary = Window.partitionBy("dept").orderBy(F.desc("salary"))

employees.select(
    "name", "dept", "salary",
    F.row_number().over(by_dept_salary).alias("row_number"),
    F.rank().over(by_dept_salary).alias("rank"),
    F.dense_rank().over(by_dept_salary).alias("dense_rank"),
    F.round(F.percent_rank().over(by_dept_salary), 2).alias("percent_rank"),
    F.ntile(2).over(by_dept_salary).alias("half"),
).orderBy("dept", "row_number").show()

## 3. Top-N per group

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Returns the top 2 salaries per department, first with <code>dense_rank</code> (ties included) and then with <code>row_number</code> (exactly 2 rows).<br><br>
<b>Why it matters</b><br>"Top N per group" is one of the most common interview coding questions. Choose the ranking function on purpose: <code>row_number</code> for exactly N rows, <code>dense_rank</code> to include everyone tied at the boundary.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
With <code>dense_rank</code>: Data has 3 rows (Senthil, Amal, Ravi), because Amal and Ravi tie for second. With <code>row_number</code>: exactly 2 rows per department.
</div>

In [0]:
ranked = employees.withColumns({
    "dr": F.dense_rank().over(by_dept_salary),
    "rn": F.row_number().over(Window.partitionBy("dept").orderBy(F.desc("salary"), "name")),
})

print("Top 2 salary levels (ties included):")
ranked.filter("dr <= 2").select("dept", "name", "salary", "dr").orderBy("dept", "dr").show()

print("Exactly 2 people per department (name breaks ties):")
ranked.filter("rn <= 2").select("dept", "name", "salary", "rn").orderBy("dept", "rn").show()

## 4. Top-N in SQL, and QUALIFY on Databricks

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the same top-N query in SQL with a subquery, then with <code>QUALIFY</code>.<br><br>
<b>Why it matters</b><br>Window results can't be filtered in <code>WHERE</code>, because <code>WHERE</code> runs before windows are computed. Standard SQL needs a subquery or CTE. Databricks SQL supports <code>QUALIFY</code>, which filters on window results directly, like <code>HAVING</code> does for aggregates.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
<code>QUALIFY</code> is a Databricks SQL extension. It works in your Free Edition workspace, but not in open-source Apache Spark.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Both queries return the top 2 earners per department by <code>row_number</code>.
</div>

In [0]:
employees.createOrReplaceTempView("employees_v")
spark.sql("""
    SELECT dept, name, salary
    FROM (
        SELECT *, row_number() OVER (PARTITION BY dept ORDER BY salary DESC, name) AS rn
        FROM employees_v
    )
    WHERE rn <= 2
    ORDER BY dept, salary DESC
""").show()

In [0]:
spark.sql("""
    SELECT dept, name, salary
    FROM employees_v
    QUALIFY row_number() OVER (PARTITION BY dept ORDER BY salary DESC, name) <= 2
    ORDER BY dept, salary DESC
""").show()

## 5. lag and lead: compare with the previous or next row

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes month-over-month change per customer with <code>lag</code>, and the next month's value with <code>lead</code>.<br><br>
<b>Why it matters</b><br>Period-over-period comparisons, churn detection ("no purchase next month") and change-data detection all use <code>lag</code>/<code>lead</code>. The first row of each partition has no previous row, so <code>lag</code> returns null there (or a default you supply).
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
For C1: <code>prev_amount</code> is null in January, then 100, 120, 90. <code>change_pct</code> for February is 20.0. <code>next_amount</code> is null in the last month of each customer.
</div>

In [0]:
by_cust_month = Window.partitionBy("customer_id").orderBy("month")

mom = revenue.select(
    "customer_id", "month", "amount",
    F.lag("amount", 1).over(by_cust_month).alias("prev_amount"),
    F.lead("amount", 1).over(by_cust_month).alias("next_amount"),
    F.lag("amount", 1, 0.0).over(by_cust_month).alias("prev_or_zero"),
).withColumn(
    "change_pct", F.round((F.col("amount") - F.col("prev_amount")) / F.col("prev_amount") * 100, 1)
)
mom.orderBy("customer_id", "month").show()

## 6. Running totals and moving averages with frames

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes a running total, a 3-month moving average and each customer's first month, using explicit <code>rowsBetween</code> frames.<br><br>
<b>Why it matters</b><br>The frame decides which rows the aggregate sees. <code>unboundedPreceding</code> to <code>currentRow</code> gives a running total. <code>-2</code> to <code>0</code> gives "this row and the two before it".
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
C1 running total: 100, 220, 310, 460. C1 3-month moving average in April: (120 + 90 + 150) / 3 = 120.0. <code>first_month</code> is the customer's earliest month on every row.
</div>

In [0]:
running = by_cust_month.rowsBetween(Window.unboundedPreceding, Window.currentRow)
last_3 = by_cust_month.rowsBetween(-2, Window.currentRow)
whole = by_cust_month.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

revenue.select(
    "customer_id", "month", "amount",
    F.sum("amount").over(running).alias("running_total"),
    F.round(F.avg("amount").over(last_3), 1).alias("moving_avg_3"),
    F.first("month").over(whole).alias("first_month"),
    F.last("month").over(whole).alias("last_month"),
).orderBy("customer_id", "month").show()

## 7. The default frame trap

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Uses <code>last()</code> and <code>sum()</code> with only <code>orderBy</code> (no explicit frame), and compares with an explicit full frame.<br><br>
<b>Why it matters</b><br>When a window has <code>orderBy</code> but no frame, Spark uses <code>RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW</code>. So <code>last()</code> returns the <b>current</b> row's value, not the partition's last value. And with <code>range</code> frames, <b>ties in the order column</b> are included together. This surprises almost everyone once.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>last_default</code> equals the current row's month on every row (not the final month). For C3, which has two rows with amount 75.0, <code>sum_by_amount_default</code> includes <b>both</b> tied rows at once, while the <code>rows</code> frame adds them one at a time.
</div>

In [0]:
revenue.filter("customer_id = 'C3'").select(
    "month", "amount",
    F.last("month").over(by_cust_month).alias("last_default"),
    F.last("month").over(whole).alias("last_full_frame"),
    F.sum("amount").over(Window.partitionBy("customer_id").orderBy("amount")).alias("sum_by_amount_default"),
    F.sum("amount").over(Window.partitionBy("customer_id").orderBy("amount")
                         .rowsBetween(Window.unboundedPreceding, Window.currentRow)).alias("sum_by_amount_rows"),
).orderBy("amount", "month").show()

## 8. rangeBetween: frames based on values

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Computes a rolling 3-month sum where the frame is defined by the <b>month number</b>, not the row count, so a missing month (C2 has no March) is handled correctly.<br><br>
<b>Why it matters</b><br><code>rowsBetween(-2, 0)</code> means "the 2 previous rows", which is wrong when months are missing. <code>rangeBetween(-2, 0)</code> on a numeric month index means "the 2 previous <b>months</b>".
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
For C2 in April: <code>rows_3</code> is 890.0 because it includes January, February and April (the 2 previous <b>rows</b>). <code>range_3</code> is 590.0, because it includes only February and April: March is missing, and January is more than 2 months earlier.
</div>

In [0]:
with_idx = revenue.withColumn(
    "month_idx", F.year(F.to_date("month", "yyyy-MM")) * 12 + F.month(F.to_date("month", "yyyy-MM"))
)
by_idx = Window.partitionBy("customer_id").orderBy("month_idx")

with_idx.filter("customer_id = 'C2'").select(
    "month", "amount",
    F.sum("amount").over(by_idx.rowsBetween(-2, 0)).alias("rows_3"),
    F.sum("amount").over(by_idx.rangeBetween(-2, 0)).alias("range_3"),
).orderBy("month").show()

## 9. Gaps and islands: find consecutive streaks

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Finds streaks of consecutive months for each customer with the "row_number difference" trick.<br><br>
<b>Why it matters</b><br>A classic advanced interview question: for consecutive values, <code>month_idx - row_number</code> is constant within a streak. Group by that constant to get each island.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
C1: one streak of 4 months. C2: two streaks (Jan–Feb and Apr). C3: one streak of 5 months.
</div>

In [0]:
islands = with_idx.withColumn(
    "grp", F.col("month_idx") - F.row_number().over(by_idx)
)
islands.groupBy("customer_id", "grp").agg(
    F.min("month").alias("streak_start"),
    F.max("month").alias("streak_end"),
    F.count("*").alias("months"),
).drop("grp").orderBy("customer_id", "streak_start").show()

## Under the hood

```
F.row_number().over(Window.partitionBy("dept").orderBy("salary"))

  Exchange hashpartitioning(dept)          <- shuffle: all rows of a dept to one partition
  Sort [dept, salary]                      <- sort inside each partition
  Window [row_number() ...]                <- one pass over the sorted rows
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> a window with `partitionBy` shuffles by the partition columns, then sorts.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> lazy like everything else. Several windows with the **same** spec share one shuffle and sort.

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ No partitionBy = one giant partition</b><br>
A window <b>without</b> <code>partitionBy</code> (for example a global <code>row_number()</code> over <code>orderBy("id")</code>) moves <b>all</b> data into a single partition on one executor. Spark logs a warning, and on big data this is slow or fails. Always partition, or use <code>monotonically_increasing_id()</code> if you only need unique (not consecutive) IDs.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The first plan shows <code>Exchange hashpartitioning(dept...)</code>, <code>Sort</code> and <code>Window</code>. The second shows <code>Exchange SinglePartition</code>, meaning all data goes to one task.
</div>

In [0]:
data = spark.range(1_000_000).withColumn("dept", F.col("id") % 10)

print("---- partitioned window ----")
data.withColumn("rn", F.row_number().over(Window.partitionBy("dept").orderBy("id"))).explain()

print("---- window without partitionBy ----")
data.withColumn("rn", F.row_number().over(Window.orderBy("id"))).explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>last()</code> returns the current row's value</b><br>
The default frame ends at the current row. Use <code>rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)</code>.<br><br>
<b>⛔ Problem: running total jumps by two rows at once</b><br><code>orderBy</code> has ties, and the default <code>RANGE</code> frame includes all tied rows. Use <code>rowsBetween</code> and a unique tie-breaker.<br><br>
<b>⛔ Problem: "No Partition Defined for Window operation" warning, slow job</b><br>The window has no <code>partitionBy</code>, so all data goes to one task. Add a partition key.<br><br>
**⛔ Problem: <code>row_number</code> gives different results each run**<br>Ties in the order columns. Add a tie-breaker.<br><br>
**⛔ Problem: can't use a window function in <code>WHERE</code>**<br>Windows are computed after <code>WHERE</code>. Use a subquery/CTE, <code>QUALIFY</code> on Databricks, or <code>withColumn</code> then <code>filter</code> in PySpark.<br><br>
<b>⛔ Problem: moving average is wrong when periods are missing</b><br><code>rowsBetween</code> counts rows, not time. Use <code>rangeBetween</code> on a numeric time index, or fill missing periods first.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is a window function and how is it different from <code>groupBy</code>?</b><br>
It computes a value for each row using a set of related rows, defined by partition, order and frame. <code>groupBy</code> collapses each group to one row, while a window function keeps all rows and adds the computed value next to them.<br><br>

<b>🎤 2. What is the difference between <code>row_number</code>, <code>rank</code> and <code>dense_rank</code>?</b><br>
For ties: <code>row_number</code> always gives unique numbers (1, 2, 3, 4), <code>rank</code> gives tied rows the same number and skips the next ones (1, 2, 2, 4), and <code>dense_rank</code> gives tied rows the same number without gaps (1, 2, 2, 3).<br><br>

<b>🎤 3. How do you get the top 3 products per category?</b><br>
<code>dense_rank</code> or <code>row_number</code> over <code>Window.partitionBy("category").orderBy(desc("sales"))</code>, then filter where the rank is at most 3. <code>row_number</code> gives exactly 3, <code>dense_rank</code> includes ties. In Databricks SQL I can use <code>QUALIFY</code>.<br><br>

<b>🎤 4. How do you keep only the latest record per key?</b><br>
<code>row_number()</code> over a window partitioned by the key and ordered by the timestamp descending with a tie-breaker, then filter where it equals 1.<br><br>

<b>🎤 5. What is the default window frame?</b><br>
With <code>orderBy</code> and no frame, it is <code>RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW</code>. Without <code>orderBy</code>, it is the whole partition. That's why <code>last()</code> with only <code>orderBy</code> returns the current row.<br><br>

<b>🎤 6. What is the difference between <code>rowsBetween</code> and <code>rangeBetween</code>?</b><br>
<code>rowsBetween</code> counts physical rows before and after the current row. <code>rangeBetween</code> uses the values of the order column, so all rows within a value range are included, which handles ties and missing periods.<br><br>

<b>🎤 7. How does Spark execute a window function?</b><br>
It shuffles the data by the partition columns, sorts each partition by the order columns, then computes the function in one pass. Windows with the same spec share the shuffle. A window without <code>partitionBy</code> moves all data to a single partition.<br><br>

<b>🎤 8. How do you compute month-over-month growth?</b><br>
<code>lag("amount").over(Window.partitionBy("customer").orderBy("month"))</code>, then <code>(amount - prev) / prev</code>. The first month has no previous value, so it's null unless I give <code>lag</code> a default.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Scores in one partition, ordered descending, are 95, 90, 90, 85. What does <code>dense_rank()</code> return for 85?</b><br>
A. 2<br>
B. 3<br>
C. 4<br>
D. 5<br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>dense_rank</code> gives 1, 2, 2, 3. <code>rank</code> would give 4.</details><br><br>

<b>Q2. Which clause filters rows based on the result of a window function in Databricks SQL without a subquery?</b><br>
A. <code>HAVING</code><br>
B. <code>WHERE</code><br>
C. <code>QUALIFY</code><br>
D. <code>FILTER</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b> <code>QUALIFY</code> is evaluated after window functions. <code>WHERE</code> runs before them, and <code>HAVING</code> filters aggregates.</details><br><br>

<b>Q3. A data engineer must keep only the most recent row per <code>device_id</code> based on <code>event_ts</code>. Which approach is correct?</b><br>
A. <code>groupBy("device_id").agg(max("event_ts"))</code> alone<br>
B. <code>row_number()</code> over a window partitioned by <code>device_id</code> ordered by <code>event_ts DESC</code>, then keep rank 1<br>
C. <code>dropDuplicates(["device_id"])</code> after <code>orderBy("event_ts")</code><br>
D. <code>lead("event_ts")</code> over a window partitioned by <code>device_id</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> A returns only the key and timestamp, not the full row. C doesn't guarantee which row is kept.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>revenue</code> and <code>employees</code>:
<ol><li>For each country, rank customers by their <b>total</b> revenue (aggregate first, then rank)</li><li>For each customer, flag months where revenue <b>dropped</b> compared to the previous month</li><li>Compute each customer's cumulative revenue and the percentage of their lifetime total reached so far</li><li>For each employee, show the salary gap to the next-higher salary in their department (null for the top earner)</li><li>Find each customer's best month (highest amount), keeping ties</li><li>Explain in a comment why <code>F.last("amount").over(Window.partitionBy("customer_id").orderBy("month"))</code> doesn't return the last month's amount</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Aggregate, then rank within country
totals = revenue.groupBy("country", "customer_id").agg(F.sum("amount").alias("total"))
totals.withColumn("rank_in_country", F.dense_rank().over(Window.partitionBy("country").orderBy(F.desc("total")))).show()

# 2. Drop vs previous month
w = Window.partitionBy("customer_id").orderBy("month")
revenue.withColumn("dropped", F.col("amount") < F.lag("amount").over(w)) \
       .orderBy("customer_id", "month").show()

# 3. Cumulative and % of lifetime total
cum = w.rowsBetween(Window.unboundedPreceding, Window.currentRow)
revenue.select(
    "customer_id", "month", "amount",
    F.sum("amount").over(cum).alias("cumulative"),
    F.round(F.sum("amount").over(cum) / F.sum("amount").over(Window.partitionBy("customer_id")) * 100, 1).alias("pct_of_lifetime"),
).orderBy("customer_id", "month").show()

# 4. Gap to the next-higher salary: lag over salary descending gives the higher earner
desc_w = Window.partitionBy("dept").orderBy(F.desc("salary"), "name")
employees.withColumn("gap_to_next_higher", F.lag("salary").over(desc_w) - F.col("salary")).show()

# 5. Best month with ties
revenue.withColumn("r", F.rank().over(Window.partitionBy("customer_id").orderBy(F.desc("amount")))) \
       .filter("r = 1").drop("r").show()

# 6. With orderBy and no frame, the default frame is RANGE UNBOUNDED PRECEDING .. CURRENT ROW,
#    so last() sees rows only up to the current one and returns the current row's value.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Windows compute per row over related rows. Rows are <b>not</b> collapsed</li><li>Spec = <code>partitionBy</code> + <code>orderBy</code> + frame (<code>rowsBetween</code> / <code>rangeBetween</code>)</li><li><code>row_number</code> (unique), <code>rank</code> (gaps), <code>dense_rank</code> (no gaps)</li><li><code>lag</code>/<code>lead</code> for previous/next values. The first/last row gives null unless you pass a default</li><li>Running totals: <code>rowsBetween(unboundedPreceding, currentRow)</code>. Moving windows: <code>rowsBetween(-n, 0)</code></li><li>Default frame with <code>orderBy</code> ends at the <b>current row</b> (<code>last()</code> trap) and includes ties (<code>RANGE</code>)</li><li><code>rangeBetween</code> on a numeric time index handles missing periods</li><li>Execution: shuffle by partition columns, sort, one pass. No <code>partitionBy</code> means a single partition</li><li>Filter window results with a subquery, <code>withColumn</code> + <code>filter</code>, or <code>QUALIFY</code> (Databricks)</li></ul>
</div>

| Task | Code |
|---|---|
| Spec | `Window.partitionBy("k").orderBy(F.desc("v"))` |
| Rank | `F.row_number()`, `F.rank()`, `F.dense_rank()` `.over(w)` |
| Previous / next | `F.lag("v", 1, default).over(w)`, `F.lead("v").over(w)` |
| Running total | `F.sum("v").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow))` |
| Moving avg | `F.avg("v").over(w.rowsBetween(-2, 0))` |
| Whole partition | `w.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)` |
| Top N | `withColumn("r", F.row_number().over(w)).filter("r <= N")` |

## Git commit

```
git add 01_spark_basics/19_window_functions.ipynb
git commit -m "add 19 window functions notebook"
```